In [1]:
!pip install -q transformers safetensors huggingface_hub

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import safetensors.torch as st
from dataclasses import dataclass
from huggingface_hub import hf_hub_download
from transformers import GPT2Tokenizer, GPT2LMHeadModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('Device :', device)

PyTorch: 2.11.0+cu130
Device : cuda


In [3]:
@dataclass
class GPTConfig:
    vocab_size: int = 50257
    context_length: int = 1024
    emb_dim: int = 768
    n_heads: int = 12
    n_layers: int = 12
    drop_rate: float = 0.1

cfg = GPTConfig()
print(cfg)

GPTConfig(vocab_size=50257, context_length=1024, emb_dim=768, n_heads=12, n_layers=12, drop_rate=0.1)


In [4]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

text = 'Alan Turing was a brilliant mathematician.'
ids = tokenizer.encode(text)

print('Text   :', text)
print('IDs    :', ids)
print('Tokens :', [tokenizer.decode([i]) for i in ids])

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Text   : Alan Turing was a brilliant mathematician.
IDs    : [36235, 39141, 373, 257, 10457, 48251, 13]
Tokens : ['Alan', ' Turing', ' was', ' a', ' brilliant', ' mathematician', '.']


In [5]:
class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.emb_dim % cfg.n_heads == 0
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.emb_dim // cfg.n_heads
        self.drop_rate = cfg.drop_rate
        self.qkv = nn.Linear(cfg.emb_dim, 3 * cfg.emb_dim)
        self.proj = nn.Linear(cfg.emb_dim, cfg.emb_dim)
        self.resid_drop = nn.Dropout(cfg.drop_rate)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q, k, v = (t.view(B, T, self.n_heads, self.head_dim).transpose(1, 2) for t in (q, k, v))
        out = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.drop_rate if self.training else 0.0,
            is_causal=True,
        )
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.resid_drop(self.proj(out))


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.emb_dim, 4 * cfg.emb_dim)
        self.act = nn.GELU(approximate='tanh')
        self.proj = nn.Linear(4 * cfg.emb_dim, cfg.emb_dim)
        self.drop = nn.Dropout(cfg.drop_rate)

    def forward(self, x):
        return self.drop(self.proj(self.act(self.fc(x))))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.emb_dim)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.emb_dim)
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

In [6]:
class GPT2(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.emb_dim)
        self.pos_emb = nn.Embedding(cfg.context_length, cfg.emb_dim)
        self.drop = nn.Dropout(cfg.drop_rate)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.emb_dim)
        self.head = nn.Linear(cfg.emb_dim, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape
        assert T <= self.cfg.context_length
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln_f(x))


model = GPT2(cfg)
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,} ({n_params / 1e6:.2f}M)')

with torch.no_grad():
    out = model(torch.tensor([[100, 200, 300, 400]]))
print('Logits shape:', tuple(out.shape))

Parameters: 124,439,808 (124.44M)
Logits shape: (1, 4, 50257)


In [7]:
@torch.no_grad()
def generate(model, input_ids, max_new_tokens, temperature=0.0, top_k=None):
    model.eval()
    ctx = model.cfg.context_length
    for _ in range(max_new_tokens):
        logits = model(input_ids[:, -ctx:])[:, -1, :]
        if temperature == 0.0:
            next_id = logits.argmax(dim=-1, keepdim=True)
        else:
            logits = logits / temperature
            if top_k is not None:
                kth = torch.topk(logits, min(top_k, logits.size(-1))).values[:, -1, None]
                logits = logits.masked_fill(logits < kth, float('-inf'))
            next_id = torch.multinomial(torch.softmax(logits, dim=-1), num_samples=1)
        input_ids = torch.cat([input_ids, next_id], dim=-1)
    return input_ids


def complete(model, prompt, **kwargs):
    ids = torch.tensor(tokenizer.encode(prompt), device=device).unsqueeze(0)
    return tokenizer.decode(generate(model, ids, **kwargs)[0].tolist())

In [8]:
model = model.to(device)
print(complete(model, 'The meaning of life is', max_new_tokens=15))

The meaning of life is is is is is is is is is is is is is is is is


In [9]:
def load_openai_weights(model):
    path = hf_hub_download('gpt2', 'model.safetensors')
    src = st.load_file(path)
    dst = model.state_dict()

    table = {
        'tok_emb.weight': 'wte.weight',
        'pos_emb.weight': 'wpe.weight',
        'ln_f.weight': 'ln_f.weight',
        'ln_f.bias': 'ln_f.bias',
    }
    pairs = [
        ('ln1', 'ln_1'), ('ln2', 'ln_2'),
        ('attn.qkv', 'attn.c_attn'), ('attn.proj', 'attn.c_proj'),
        ('mlp.fc', 'mlp.c_fc'), ('mlp.proj', 'mlp.c_proj'),
    ]
    for i in range(model.cfg.n_layers):
        for ours, theirs in pairs:
            for kind in ('weight', 'bias'):
                table[f'blocks.{i}.{ours}.{kind}'] = f'h.{i}.{theirs}.{kind}'

    transposed = ('attn.qkv', 'attn.proj', 'mlp.fc', 'mlp.proj')
    with torch.no_grad():
        for ours, theirs in table.items():
            tensor = src[theirs]
            if any(f'.{m}.weight' in ours for m in transposed):
                tensor = tensor.t()
            dst[ours].copy_(tensor)

    print(f'Copied {len(table)} tensors')
    return model


gpt2 = GPT2(cfg)
gpt2 = load_openai_weights(gpt2).to(device).eval()

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Copied 148 tensors


In [10]:
reference = GPT2LMHeadModel.from_pretrained('gpt2').to(device).eval()

probe = torch.tensor([tokenizer.encode('The quick brown fox jumps over the lazy dog')], device=device)
with torch.no_grad():
    mine = gpt2(probe)
    theirs = reference(probe).logits

print('Max abs logit difference:', (mine - theirs).abs().max().item())
print('Same argmax at every position:', torch.equal(mine.argmax(-1), theirs.argmax(-1)))

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Max abs logit difference: 6.866455078125e-05
Same argmax at every position: True


In [11]:
torch.manual_seed(42)

prompts = [
    'Alan Turing was a',
    'The theory of general relativity states that',
    'In a distant galaxy, astronomers discovered',
    'To make a delicious cup of coffee, first',
]

for p in prompts:
    print('Prompt:', p)
    print('Output:', complete(gpt2, p, max_new_tokens=35, temperature=0.8, top_k=40))
    print('-' * 80)

Prompt: Alan Turing was a
Output: Alan Turing was a successful inventor of the first Turing machine, and had been able to synthesize it into a human being with the help of the computer himself, but this was not what he was
--------------------------------------------------------------------------------
Prompt: The theory of general relativity states that
Output: The theory of general relativity states that in each case, there are two things that must be different:

The first is that the world must be moving. The second is that the universe must be stationary and
--------------------------------------------------------------------------------
Prompt: In a distant galaxy, astronomers discovered
Output: In a distant galaxy, astronomers discovered the first signs of the Universe.

A new study from the University of California — Davis and the California Institute of Technology indicates that galaxies have a massive mass, but are
--------------------------------------------------------------

In [12]:
torch.manual_seed(42)

prompt = 'Once upon a time in a small village nestled between the mountains,'
settings = [
    ('Greedy (T=0)', dict(temperature=0.0)),
    ('Balanced (T=0.8, top_k=40)', dict(temperature=0.8, top_k=40)),
    ('Diverse (T=1.2, top_k=50)', dict(temperature=1.2, top_k=50)),
]

for name, kwargs in settings:
    print(f'[{name}]')
    print(complete(gpt2, prompt, max_new_tokens=30, **kwargs))
    print('=' * 70)

[Greedy (T=0)]
Once upon a time in a small village nestled between the mountains, a young girl named Nana was born.

Nana was a girl who had been raised by her father, and had been raised by her
[Balanced (T=0.8, top_k=40)]
Once upon a time in a small village nestled between the mountains, a wild beast that had been known as the "Easter Cane" had emerged from within. The beast had a head with a long neck and
[Diverse (T=1.2, top_k=50)]
Once upon a time in a small village nestled between the mountains, one hundred years ago a woman named Kala called him, "Boomkukaht!" With some effort, she helped to create a village




| Aspect | Vaswani et al. (2017) | GPT-2 (2019) |
| :--- | :--- | :--- |
| Architecture | Encoder-decoder | Decoder-only |
| Positional encoding | Fixed sinusoidal | Learned embeddings |
| Layer norm | Post-LN | Pre-LN plus final LN |
| Attention | Bidirectional encoder, causal decoder | Causal everywhere |
| Weight tying | Optional | Embedding tied to output head |
| Activation | ReLU | GELU (tanh) |

The from-scratch model reproduces the official GPT-2 (124M) outputs after loading the pretrained weights.